In [2]:
import fitz
from tqdm.auto import tqdm
import os

pdf_path = "../dataset/human-nutrition-text.pdf"

def text_formatter(text: str) -> str:
    """Performs minor formatting on text."""
    
    cleaned_text = text.replace("\n", " ").strip()
    return cleaned_text

def open_and_read_pdf(pdf_path: str) -> list[dict]:
    """
    Opens a PDF file, reads its text content page by page, and collects statistics.

    Parameters:
        pdf_path (str): The file path to the PDF document to be opened and read.

    Returns:
        list[dict]: A list of dictionaries, each containing the page number
        (adjusted), character count, word count, sentence count, token count, and the extracted text
        for each page.
    """
    doc = fitz.open(pdf_path)  # open a document
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc)):  # iterate the document pages
        text = page.get_text()  # get plain text encoded as UTF-8
        text = text_formatter(text)
        pages_and_texts.append({"page_number": page_number - 14,  # adjust page numbers since our PDF starts on page 42
                                "page_char_count": len(text),
                                "page_word_count": len(text.split(" ")),
                                "page_sentence_count_raw": len(text.split(". ")),
                                "page_token_count": len(text) / 4,  # 1 token = ~4 chars
                                "text": text})
    return pages_and_texts

if not os.path.exists(pdf_path):
    print("[INFO] File doesn't exist")
else:
    pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)

0it [00:00, ?it/s]

In [3]:
pages_and_texts[:2]

[{'page_number': -14,
  'page_char_count': 29,
  'page_word_count': 4,
  'page_sentence_count_raw': 1,
  'page_token_count': 7.25,
  'text': 'Human Nutrition: 2020 Edition'},
 {'page_number': -13,
  'page_char_count': 0,
  'page_word_count': 1,
  'page_sentence_count_raw': 1,
  'page_token_count': 0.0,
  'text': ''}]

In [4]:
pages_and_texts[1044]

{'page_number': 1030,
 'page_char_count': 1308,
 'page_word_count': 221,
 'page_sentence_count_raw': 13,
 'page_token_count': 327.0,
 'text': 'Giardia lamblia is another parasite that is found in contaminated  drinking water. In addition, it lives in the intestinal tracts of animals,  and can wash into surface water and reservoirs, similar to  Cryptosporidium. Giardia causes giardiasis, with symptoms that  include abdominal cramping and diarrhea within one to three days.  Although most people recover within one to two weeks, the disease  can lead to a chronic condition, especially in people with  compromised immune systems.  The  parasite  Toxoplasma  gondii  causes  the  infection  toxoplasmosis, which is a leading cause of death attributed to  foodborne illness in the United States. More than sixty million  Americans carry Toxoplasma gondii, but very few have symptoms.  Typically, the body’s immune system keeps the parasite from  causing disease. Sources include raw or undercooked me

In [5]:
import random

random.sample(pages_and_texts, k=3)

[{'page_number': 539,
  'page_char_count': 0,
  'page_word_count': 1,
  'page_sentence_count_raw': 1,
  'page_token_count': 0.0,
  'text': ''},
 {'page_number': 554,
  'page_char_count': 1650,
  'page_word_count': 288,
  'page_sentence_count_raw': 16,
  'page_token_count': 412.5,
  'text': 'Dietary Reference Intakes for Vitamin A  There is more than one source of vitamin A in the diet. There is  preformed vitamin A, which is abundant in many animal-derived  foods, and there are carotenoids, which are found in high  concentrations in vibrantly colored fruits and vegetables and some  oils.  Some carotenoids are converted to retinol in the body by  intestinal cells and liver cells. However, only minuscule amounts  of certain carotenoids are converted to retinol, meaning fruits and  vegetables are not necessarily good sources of vitamin A.  The RDA for vitamin A includes all sources of vitamin A. The  RDA for vitamin A is given in mcg of Retinol Activity Equivalent  (RAE) to take into acco

In [6]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
df

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,-14,29,4,1,7.25,Human Nutrition: 2020 Edition
1,-13,0,1,1,0.00,
2,-12,320,54,1,80.00,Human Nutrition: 2020 Edition UNIVERSITY OF ...
3,-11,212,32,1,53.00,Human Nutrition: 2020 Edition by University of...
4,-10,797,145,2,199.25,Contents Preface University of Hawai‘i at Mā...
...,...,...,...,...,...,...
1203,1189,1676,252,18,419.00,39. Exercise 10.2 & 11.3 reused “Egg Oval Food...
1204,1190,1617,254,10,404.25,Images / Pixabay License; “Pumpkin Cartoon Ora...
1205,1191,1715,261,13,428.75,Flashcard Images Note: Most images in the fla...
1206,1192,1733,268,13,433.25,ShareAlike 11. Organs reused “Pancreas Organ ...


In [7]:
# Get stats
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count
count,1208.00,1208.00,1208.00,1208.00,1208.00
mean,589.50,1148.59,198.89,9.97,287.15
std,348.86,560.44,95.75,6.19,140.11
min,-14.00,0.00,1.00,1.00,0.00
25%,287.75,762.75,134.00,4.00,190.69
50%,589.50,1232.50,215.00,10.00,308.12
75%,891.25,1605.25,271.25,14.00,401.31
max,1193.00,2308.00,429.00,32.00,577.00


In [17]:
from spacy.lang.en import English

nlp = English()

nlp.add_pipe("sentencizer")

for item in tqdm(pages_and_texts):
    item["sentences"] = list(nlp(item["text"]).sents)
    
    # Make sure all sentences are strings
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]
    
    # Count the sentences 
    item["page_sentence_count_spacy"] = len(item["sentences"])

  0%|          | 0/1208 [00:00<?, ?it/s]

In [18]:
random.sample(pages_and_texts, k=1)

[{'page_number': 1113,
  'page_char_count': 955,
  'page_word_count': 161,
  'page_sentence_count_raw': 7,
  'page_token_count': 238.75,
  'text': 'Image by  Allison  Calabrese /  CC BY 4.0  fiber intake because of what the breakdown products of the fiber  do for the colon. The bacterial breakdown of fiber in the large  intestine releases short-chain fatty acids. These molecules have  been found to nourish colonic cells, inhibit colonic inflammation,  and stimulate the immune system (thereby providing protection  of the colon from harmful substances). Additionally, the bacterial  indigestible fiber, mostly insoluble, increases stool bulk and softness  increasing transit time in the large intestine and facilitating feces  elimination. One phenomenon of consuming foods high in fiber is  increased gas, since the byproducts of bacterial digestion of fiber  are gases.  Figure 18.2 Diverticulitis: A Disease of Fiber Deficiency  Some studies have found a link between high dietary-fiber intake

In [19]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,589.50,1148.59,198.89,9.97,287.15,10.32
std,348.86,560.44,95.75,6.19,140.11,6.30
min,-14.00,0.00,1.00,1.00,0.00,0.00
25%,287.75,762.75,134.00,4.00,190.69,5.00
50%,589.50,1232.50,215.00,10.00,308.12,10.00
75%,891.25,1605.25,271.25,14.00,401.31,15.00
max,1193.00,2308.00,429.00,32.00,577.00,28.00


# Chuncking

In [20]:
# Define split size to turn groups of sentences into chunks
num_sentence_chunk_size = 10 

# Create a function that recursively splits a list into desired sizes
def split_list(input_list: list, 
               slice_size: int) -> list[list[str]]:
    """
    Splits the input_list into sublists of size slice_size (or as close as possible).

    For example, a list of 17 sentences would be split into two lists of [[10], [7]]
    """
    return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]

# Loop through pages and texts and split sentences into chunks
for item in tqdm(pages_and_texts):
    item["sentence_chunks"] = split_list(input_list=item["sentences"],
                                         slice_size=num_sentence_chunk_size)
    item["num_chunks"] = len(item["sentence_chunks"])

  0%|          | 0/1208 [00:00<?, ?it/s]

In [21]:
# Sample an example from the group (note: many samples have only 1 chunk as they have <=10 sentences total)
random.sample(pages_and_texts, k=1)

[{'page_number': 759,
  'page_char_count': 711,
  'page_word_count': 149,
  'page_sentence_count_raw': 1,
  'page_token_count': 177.75,
  'text': 'Image by  USDA/  USDA  Organic  Term  Explanation  Lean  Fewer than a set amount of grams of fat for that  particular cut of meat or seafood  High, Rich In,  or Excellent  Source Of  Contains 20% or more of the nutrient’s DV  Good source,  Contains or  Provides  Contains 10 to 19% of the nutrient’s DV  Light/lite  Contains ⅓ fewer calories or 50% less fat; if more than  half of calories come from fat, then fat content must be  reduced by 50% or more  Organic1  Contains 95% certified organic ingredients  1 The term “Organic” is regulated by the USDA and appears as a  USDA Organic Seal in the front of packaged food products,  beverages and dietary supplements  Source:  732  |  Discovering Nutrition Facts',
  'sentences': ['Image by  USDA/  USDA  Organic  Term  Explanation  Lean  Fewer than a set amount of grams of fat for that  particular cut 

In [22]:
# Create a DataFrame to get stats
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy,num_chunks
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,589.50,1148.59,198.89,9.97,287.15,10.32,1.53
std,348.86,560.44,95.75,6.19,140.11,6.30,0.64
min,-14.00,0.00,1.00,1.00,0.00,0.00,0.00
25%,287.75,762.75,134.00,4.00,190.69,5.00,1.00
50%,589.50,1232.50,215.00,10.00,308.12,10.00,1.00
75%,891.25,1605.25,271.25,14.00,401.31,15.00,2.00
max,1193.00,2308.00,429.00,32.00,577.00,28.00,3.00


# Splitting Chunks

In [23]:
import re

# Split each chunk into its own item
pages_and_chunks = []
for item in tqdm(pages_and_texts):
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}
        chunk_dict["page_number"] = item["page_number"]
        
        # Join the sentences together into a paragraph-like structure, aka a chunk (so they are a single string)
        joined_sentence_chunk = "".join(sentence_chunk).replace("  ", " ").strip()
        joined_sentence_chunk = re.sub(r'\.([A-Z])', r'. \1', joined_sentence_chunk) # ".A" -> ". A" for any full-stop/capital letter combo 
        chunk_dict["sentence_chunk"] = joined_sentence_chunk

        # Get stats about the chunk
        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len([word for word in joined_sentence_chunk.split(" ")])
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4 # 1 token = ~4 characters
        
        pages_and_chunks.append(chunk_dict)

# How many chunks do we have?
len(pages_and_chunks)

  0%|          | 0/1208 [00:00<?, ?it/s]

1843

In [24]:
# View a random sample
random.sample(pages_and_chunks, k=1)

[{'page_number': 1061,
  'sentence_chunk': 'Store food products separately in the refrigerator. • Cook. Heat food to proper temperatures. Use a food thermometer to check the temperature of food while it is cooking. Keep food hot after it has been cooked. • Chill. Refrigerate any leftovers within two hours. Never thaw or marinate food on the counter. Know when to keep food and when to throw it out. It can be helpful 1.',
  'chunk_char_count': 377,
  'chunk_word_count': 68,
  'chunk_token_count': 94.25}]

In [25]:
df = pd.DataFrame(pages_and_chunks)
df.describe().round(2)

,page_number,chunk_char_count,chunk_word_count,chunk_token_count
count,1843.00,1843.00,1843.00,1843.00
mean,610.38,734.83,112.72,183.71
std,347.79,447.43,71.07,111.86
min,-14.00,12.00,3.00,3.00
25%,307.50,315.00,45.00,78.75
50%,613.00,746.00,114.00,186.50
75%,917.00,1118.50,173.00,279.62
max,1193.00,1831.00,297.00,457.75


In [26]:
# Show random chunks with under 30 tokens in length
min_token_length = 30
for row in df[df["chunk_token_count"] <= min_token_length].sample(5).iterrows():
    print(f'Chunk token count: {row[1]["chunk_token_count"]} | Text: {row[1]["sentence_chunk"]}')

Chunk token count: 16.25 | Text: Table 14.2  Micronutrient Levels during Puberty 886 | Adolescence
Chunk token count: 20.25 | Text: Honor your health – gentle nutrition       Calories In Versus Calories Out | 1075
Chunk token count: 19.5 | Text: http:/ /pressbooks.oer.hawaii.edu/ humannutrition2/?p=519   Introduction | 991
Chunk token count: 19.25 | Text: 2018). Centers for Disease Control and 998 | The Causes of Food Contamination
Chunk token count: 6.5 | Text: Fat-Soluble Vitamins | 539


In [27]:
pages_and_chunks_over_min_token_len = df[df["chunk_token_count"] > min_token_length].to_dict(orient="records")
pages_and_chunks_over_min_token_len[:2]

[{'page_number': -12,
  'sentence_chunk': 'Human Nutrition: 2020 Edition UNIVERSITY OF HAWAI‘I AT MĀNOA FOOD SCIENCE AND HUMAN NUTRITION PROGRAM ALAN TITCHENAL, SKYLAR HARA, NOEMI ARCEO CAACBAY, WILLIAM MEINKE-LAU, YA-YUN YANG, MARIE KAINOA FIALKOWSKI REVILLA, JENNIFER DRAPER, GEMADY LANGFELDER, CHERYL GIBBY, CHYNA NICOLE CHUN, AND ALLISON CALABRESE',
  'chunk_char_count': 308,
  'chunk_word_count': 42,
  'chunk_token_count': 77.0},
 {'page_number': -11,
  'sentence_chunk': 'Human Nutrition: 2020 Edition by University of Hawai‘i at Mānoa Food Science and Human Nutrition Program is licensed under a Creative Commons Attribution 4.0 International License, except where otherwise noted.',
  'chunk_char_count': 210,
  'chunk_word_count': 30,
  'chunk_token_count': 52.5}]

# Embedding chunks

In [30]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2", 
                                      device="cpu")

In [31]:
%%time

embedding_model.to("cuda")

for item in tqdm(pages_and_chunks_over_min_token_len):
    item["embedding"] = embedding_model.encode(item["sentence_chunk"])

  0%|          | 0/1680 [00:00<?, ?it/s]

CPU times: total: 1min 46s
Wall time: 30.1 s


In [32]:
# Turn text chunks into a single list
text_chunks = [item["sentence_chunk"] for item in pages_and_chunks_over_min_token_len]

In [34]:
%%time

# Embed all texts in batches
text_chunk_embeddings = embedding_model.encode(text_chunks,
                                               batch_size=32,
                                               convert_to_tensor=True)

text_chunk_embeddings

CPU times: total: 45.4 s
Wall time: 14.9 s


tensor([[ 0.0674,  0.0902, -0.0051,  ..., -0.0221, -0.0232,  0.0126],
        [ 0.0552,  0.0592, -0.0166,  ..., -0.0120, -0.0103,  0.0227],
        [ 0.0280,  0.0340, -0.0206,  ..., -0.0054,  0.0213,  0.0313],
        ...,
        [ 0.0771,  0.0098, -0.0122,  ..., -0.0409, -0.0752, -0.0241],
        [ 0.1030, -0.0165,  0.0083,  ..., -0.0574, -0.0283, -0.0295],
        [ 0.0864, -0.0125, -0.0113,  ..., -0.0522, -0.0337, -0.0299]],
       device='cuda:0')

In [35]:
# Save embeddings to file
text_chunks_and_embeddings_df = pd.DataFrame(pages_and_chunks_over_min_token_len)
embeddings_df_save_path = "text_chunks_and_embeddings_df.csv"
text_chunks_and_embeddings_df.to_csv(embeddings_df_save_path, index=False)

In [36]:
# Import saved file and view
text_chunks_and_embedding_df_load = pd.read_csv(embeddings_df_save_path)
text_chunks_and_embedding_df_load.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,-12,Human Nutrition: 2020 Edition UNIVERSITY OF HA...,308,42,77.00,[ 6.74242377e-02 9.02281404e-02 -5.09549072e-...
1,-11,Human Nutrition: 2020 Edition by University of...,210,30,52.50,[ 5.52156232e-02 5.92139661e-02 -1.66167337e-...
2,-10,Contents Preface University of Hawai‘i at Māno...,766,114,191.50,[ 2.79802065e-02 3.39813977e-02 -2.06426624e-...
3,-9,Lifestyles and Nutrition University of Hawai‘i...,941,142,235.25,[ 6.82567060e-02 3.81274782e-02 -8.46853945e-...
4,-8,The Cardiovascular System University of Hawai‘...,998,152,249.50,[ 3.30264606e-02 -8.49768426e-03 9.57158674e-...


In [37]:
import random

import torch
import numpy as np 
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"

# Import texts and embedding df
text_chunks_and_embedding_df = pd.read_csv("text_chunks_and_embeddings_df.csv")

# Convert embedding column back to np.array (it got converted to string when it got saved to CSV)
text_chunks_and_embedding_df["embedding"] = text_chunks_and_embedding_df["embedding"].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))

# Convert texts and embedding df to list of dicts
pages_and_chunks = text_chunks_and_embedding_df.to_dict(orient="records")

# Convert embeddings to torch tensor and send to device (note: NumPy arrays are float64, torch tensors are float32 by default)
embeddings = torch.tensor(np.array(text_chunks_and_embedding_df["embedding"].tolist()), dtype=torch.float32).to(device)
embeddings.shape

torch.Size([1680, 768])

In [38]:
text_chunks_and_embedding_df.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,-12,Human Nutrition: 2020 Edition UNIVERSITY OF HA...,308,42,77.00,"[0.0674242377, 0.0902281404, -0.00509549072, -..."
1,-11,Human Nutrition: 2020 Edition by University of...,210,30,52.50,"[0.0552156232, 0.0592139661, -0.0166167337, -0..."
2,-10,Contents Preface University of Hawai‘i at Māno...,766,114,191.50,"[0.0279802065, 0.0339813977, -0.0206426624, 0...."
3,-9,Lifestyles and Nutrition University of Hawai‘i...,941,142,235.25,"[0.068256706, 0.0381274782, -0.00846853945, -0..."
4,-8,The Cardiovascular System University of Hawai‘...,998,152,249.50,"[0.0330264606, -0.00849768426, 0.00957158674, ..."


In [41]:
from sentence_transformers import util
# 1. Define the query
# Note: This could be anything. But since we're working with a nutrition textbook, we'll stick with nutrition-based queries.
query = "macronutrients functions"
print(f"Query: {query}")

# 2. Embed the query to the same numerical space as the text examples 
# Note: It's important to embed your query with the same model you embedded your examples with.
query_embedding = embedding_model.encode(query, convert_to_tensor=True)

# 3. Get similarity scores with the dot product (we'll time this for fun)
from time import perf_counter as timer

start_time = timer()
dot_scores = util.dot_score(a=query_embedding, b=embeddings)[0]
end_time = timer()

print(f"Time take to get scores on {len(embeddings)} embeddings: {end_time-start_time:.5f} seconds.")

# 4. Get the top-k results (we'll keep this to 5)
top_results_dot_product = torch.topk(dot_scores, k=5)
top_results_dot_product

Query: macronutrients functions
Time take to get scores on 1680 embeddings: 0.00070 seconds.


torch.return_types.topk(
values=tensor([0.6926, 0.6738, 0.6646, 0.6536, 0.6473], device='cuda:0'),
indices=tensor([42, 47, 41, 51, 46], device='cuda:0'))

In [42]:
# Define helper function to print wrapped text 
import textwrap

def print_wrapped(text, wrap_length=80):
    wrapped_text = textwrap.fill(text, wrap_length)
    print(wrapped_text)

In [43]:
print(f"Query: '{query}'\n")
print("Results:")
# Loop through zipped together scores and indicies from torch.topk
for score, idx in zip(top_results_dot_product[0], top_results_dot_product[1]):
    print(f"Score: {score:.4f}")
    # Print relevant sentence chunk (since the scores are in descending order, the most relevant chunk will be first)
    print("Text:")
    print_wrapped(pages_and_chunks[idx]["sentence_chunk"])
    # Print the page number too so we can reference the textbook further (and check the results)
    print(f"Page number: {pages_and_chunks[idx]['page_number']}")
    print("\n")

Query: 'macronutrients functions'

Results:
Score: 0.6926
Text:
Macronutrients Nutrients that are needed in large amounts are called
macronutrients. There are three classes of macronutrients: carbohydrates,
lipids, and proteins. These can be metabolically processed into cellular energy.
The energy from macronutrients comes from their chemical bonds. This chemical
energy is converted into cellular energy that is then utilized to perform work,
allowing our bodies to conduct their basic functions. A unit of measurement of
food energy is the calorie. On nutrition food labels the amount given for
“calories” is actually equivalent to each calorie multiplied by one thousand. A
kilocalorie (one thousand calories, denoted with a small “c”) is synonymous with
the “Calorie” (with a capital “C”) on nutrition food labels. Water is also a
macronutrient in the sense that you require a large amount of it, but unlike the
other macronutrients, it does not yield calories. Carbohydrates Carbohydrates
are 

In [44]:
# Get GPU available memory
import torch
gpu_memory_bytes = torch.cuda.get_device_properties(0).total_memory
gpu_memory_gb = round(gpu_memory_bytes / (2**30))
print(f"Available GPU memory: {gpu_memory_gb} GB")

Available GPU memory: 8 GB


In [46]:
torch.cuda.get_device_capability(0)[0]

8